# Encoding, Scaling, and Feature Engineering

> Prepare numeric, categorical, and date features using transformations appropriate to each type.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Understand standardization, min-max scaling, and robust scaling.
- Encode nominal and ordinal categories appropriately.
- Engineer simple date, ratio, and interaction features without leaking target information.

Scaling matters for distance- and gradient-based models (such as SVM, KNN, and logistic regression). Tree-based models are often less sensitive to monotonic scaling, although preprocessing still matters for consistency.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder

numeric = np.array([[10.], [12.], [14.], [100.]])
for name, scaler in [("standard", StandardScaler()), ("min-max", MinMaxScaler()), ("robust", RobustScaler())]:
    print(name, scaler.fit_transform(numeric).round(2).ravel())

standard [-0.63 -0.58 -0.52  1.73]
min-max [0.   0.02 0.04 1.  ]
robust [-0.12 -0.04  0.04  3.62]


## Encoding categories
One-hot encoding represents nominal categories as separate binary columns. Ordinal encoding is appropriate only when the categories have a meaningful order (for example, low < medium < high). Arbitrary integer codes can accidentally imply order.

In [2]:
df = pd.DataFrame({"city": ["Baku", "Tbilisi", "Baku", "Yerevan"], "satisfaction": ["low", "high", "medium", "high"]})
print(pd.get_dummies(df["city"], dtype=int))
ordinal = OrdinalEncoder(categories=[["low", "medium", "high"]])
print("Ordered satisfaction:", ordinal.fit_transform(df[["satisfaction"]]).ravel())

   Baku  Tbilisi  Yerevan
0     1        0        0
1     0        1        0
2     1        0        0
3     0        0        1
Ordered satisfaction: [0. 2. 1. 2.]


## Feature engineering: examples and guardrails
- Calendar features: weekday, month, time since an event.
- Ratios: spending per visit, cost per unit.
- Domain transformations: log of strictly positive, skewed values.
- Interactions: a feature combination with a defensible meaning.

Avoid features calculated after the outcome (e.g., “number of complaints after churn”). For time-dependent predictions, engineer features only from data available at the prediction timestamp.

In [3]:
orders = pd.DataFrame({"order_date": pd.to_datetime(["2026-01-03", "2026-02-14", "2026-03-21"]), "revenue": [120, 340, 95], "items": [3, 5, 1]})
orders["weekday"] = orders["order_date"].dt.day_name()
orders["month"] = orders["order_date"].dt.month
orders["revenue_per_item"] = orders["revenue"] / orders["items"]
print(orders)

  order_date  revenue  items   weekday  month  revenue_per_item
0 2026-01-03      120      3  Saturday      1              40.0
1 2026-02-14      340      5  Saturday      2              68.0
2 2026-03-21       95      1  Saturday      3              95.0


## Practice
1. Scale a numeric column using `StandardScaler`.
2. Decide whether “education level” is ordinal or nominal in your context.
3. Add a “customer tenure in years” feature and explain how you would avoid using future data.

In [4]:
tenure_months = np.array([[6.], [18.], [36.]])
print("Tenure in years:", (tenure_months/12).ravel())
print("For a real model, fit transformations only on the training partition or inside a Pipeline.")

Tenure in years: [0.5 1.5 3. ]
For a real model, fit transformations only on the training partition or inside a Pipeline.
